# Stage 3: Simulation-Based Dataset Exploration
## Project: AI-Enabled Digital Twin for Fouling-Aware Optimization of Textile Wastewater Reuse

> **Scientific Notice:** The Stage 3 dataset is simulation-derived from the mechanistic RO process engine (Topology A, 3:2 staging, 15 Toray TML20D-400 elements). Machine learning models trained on this dataset will initially learn the behaviour of the mechanistic simulator. Agreement between the ML surrogate and this dataset demonstrates surrogate fidelity, not independent validation against an industrial plant.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup paths and plot styles
sys.path.insert(0, str(Path("..").resolve() / "src"))
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

## 1. Load Generated Datasets

In [ ]:
data_dir = Path("../data/generated")
df_all = pd.read_csv(data_dir / "stage3_all_scenarios.csv")
df_feasible = pd.read_csv(data_dir / "stage3_feasible_scenarios.csv")
df_infeasible = pd.read_csv(data_dir / "stage3_infeasible_scenarios.csv")
df_ood = pd.read_csv(data_dir / "stage3_ood_scenarios.csv")

print(f"Total Scenarios Generated : {len(df_all):,d}")
print(f"Feasible Scenarios        : {len(df_feasible):,d} ({len(df_feasible)/len(df_all)*100:.1f}%)")
print(f"Infeasible Scenarios      : {len(df_infeasible):,d} ({len(df_infeasible)/len(df_all)*100:.1f}%)")
print(f"OOD Stress-Testing Scenarios : {len(df_ood):,d}")

## 2. Failure Mechanism Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
fail_counts = df_infeasible["failure_reason"].value_counts()
fail_counts.plot(kind="barh", ax=ax, color="#e74c3c", edgecolor="black")
ax.set_title("Distribution of Infeasible Operating Boundaries")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

## 3. Physical Conservation Verification: Recovery vs Concentrate TDS

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    df_feasible["overall_recovery_pct"],
    df_feasible["concentrate_tds_mgL"],
    c=df_feasible["feed_tds_mgL"],
    cmap="turbo",
    alpha=0.6,
    s=20
)
plt.colorbar(scatter, label="Feed TDS (mg/L)")
ax.set_xlabel("Overall Recovery (%)")
ax.set_ylabel("Final Concentrate TDS (mg/L)")
ax.set_title("Strict Solute Conservation Across Feasible Envelope")
plt.tight_layout()
plt.show()

## 4. Correlation Matrix: Causal vs Non-Causal Descriptors

In [ ]:
features = [
    "feed_flow_m3h", "feed_tds_mgL", "temperature_C", "stage1_pressure_bar", "stage2_pressure_bar",
    "feed_cod_mgL", "feed_pH",
    "overall_recovery_pct", "permeate_tds_mgL", "concentrate_tds_mgL", "average_flux_LMH", "SEC_kWh_m3"
]
corr = df_feasible[features].corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(cax, label="Pearson r")
ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha="left")
ax.set_yticklabels(features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)
ax.set_title("Correlation Matrix: Validating Non-Causal Variables (COD, pH)", pad=30)
plt.tight_layout()
plt.show()

## 5. Dataset Split Verification (70% Train / 15% Val / 15% Test)

In [ ]:
split_summary = df_feasible["dataset_split"].value_counts()
print("Deterministic Data Splits for Future ML Surrogate:")
for s_name, count in split_summary.items():
    print(f"  - {s_name:10s}: {count:,d} ({count/len(df_feasible)*100:.1f}%)")